In [ ]:
import torch
import clip
import textstat
from PIL import Image
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from transformers import BertTokenizer, BertModel, BertForMaskedLM
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 预先加载BERT模型和分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
predict_model = BertForMaskedLM.from_pretrained('bert-base-multilingual-cased')

# 加载NIMA模型（使用MobileNetV2）
nima_model_url = "https://tfhub.dev/google/tf2-preview/mobilenet_v2/classification/4"
nima_model = hub.KerasLayer(nima_model_url)

# 加载VADER情感分析器
sa = SentimentIntensityAnalyzer()

# NIMA图像预处理
def preprocess_image(image_path):
    """ 预处理图像以适应 NIMA 模型输入 """
    img = Image.open(image_path).resize((224, 224))
    img = np.array(img) / 255.0  # 将像素值归一化为 [0, 1]
    return img

# NIMA预测图像质量
def predict_image_quality(image_path):
    """ 使用 NIMA 模型预测图像质量分布并计算评分 """
    img = preprocess_image(image_path)
    img = np.expand_dims(img, axis=0)  # 添加批次维度
    preds = nima_model(img)  # 预测图像的评分分布
    scores = np.arange(1, 11)  # 评分范围为 1-10
    quality_score = np.sum(preds * scores) / np.sum(preds)
    return quality_score

# 处理数据文件，提取特征并保存
def process_data_file(input_file_path):
    # 读取原始 JSON 数据
    output_file_path = input_file_path.replace('.json', '_feature.json')
    
    with open(input_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 处理每个数据项并提取特征
    for idx, item in enumerate(data):
        title_text = str(item['title'])  # 使用样本中的文本字段
        tags_text = str(item['tags'])
        num_tags = len(tags_text)

        # 1. 使用BERT计算标题与标签的相似度
        with torch.no_grad():
            title_tokens = tokenizer(title_text, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
            title_embeddings = bert_model(**title_tokens).last_hidden_state.mean(dim=1).detach()

            tags_list = tags_text.split(';')
            sum_cosine_similarity = 0
            for tag in tags_list:
                tag = tag.strip()
                tags_tokens = tokenizer(tag, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
                tags_embeddings = bert_model(**tags_tokens).last_hidden_state.mean(dim=1).detach()
                cosine_similarity = torch.nn.functional.cosine_similarity(title_embeddings, tags_embeddings)
                sum_cosine_similarity += cosine_similarity
            
            if num_tags != 0:
                sum_cosine_similarity = sum_cosine_similarity / num_tags
            
            # 保存总余弦相似度和标签数量
            item['sum_cosine_similarity'] = sum_cosine_similarity.item()
            item['num_tags'] = num_tags

        # 2. 用CLIP计算图文相似度
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model, preprocess = clip.load("ViT-B/32", device=device)
        
        original_image_path = item['image_path']
        image_path = original_image_path.replace('./img/', '../dataset_xiaohongshu/xiaohongshu/img/')
        
        try:
            image = Image.open(image_path).convert('RGB')
            image_clip = preprocess(image).unsqueeze(0).to(device)
            text = clip.tokenize([title_text]).to(device)
        
            with torch.no_grad():
                text_features = model.encode_text(text).float()
                image_features = model.encode_image(image_clip).float()
            
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            similarity = (text_features.cpu().numpy() @ image_features.cpu().numpy().T)
            item['similarity'] = similarity.item()
        except Exception as e:
            print(f"Error processing image {image_path}: {e}")
            item['similarity'] = None

        # 3. 计算title文本可读性
        FOG = textstat.gunning_fog(title_text)
        ARI = textstat.automated_readability_index(title_text)
        CLI = textstat.coleman_liau_index(title_text)
        mean_readability = (FOG + ARI + CLI) / 3
        item['mean_readability'] = mean_readability

        # 4. 用NIMA模型评估图像质量
        try:
            image_quality_score = predict_image_quality(image_path)
            item['image_quality_score'] = image_quality_score
            print(f"Image Quality Score for {image_path}: {image_quality_score}")
        except Exception as e:
            print(f"Error processing {image_path} for NIMA: {e}")
            item['image_quality_score'] = None

        # 5. VADER情感分析
        sentiment_scores = sa.polarity_scores(title_text)
        item['title_sentiment'] = sentiment_scores

    # 将处理后的数据写入新的JSON文件
    with open(output_file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Saved output data with features to {output_file_path}")

# 处理多个文件
train_file_path = './train_data.json'
test_file_path = './test_data.json'
val_file_path = './val_data.json'

data_name = [train_file_path, test_file_path, val_file_path]

for file_name in data_name:
    process_data_file(file_name)
